# Task 3 — Notebook 02: Baseline climatology and Bayesian anomaly detection

This is the analytical core of Task 3. We first build a per-pixel, per-week
climatology from the 2015–2021 SMAP baseline, then score each event-year observation
against that baseline using two complementary frameworks.

## Why two anomaly frameworks?

**Frequentist z-score.** The standard approach — subtract the baseline mean, divide by
the standard deviation. Simple, widely used, and directly comparable to NASA/USDA
anomaly products. But with only 5–7 baseline observations per pixel per week, the
estimated σ is unreliable, and the implicit assumption that z ~ N(0,1) is generous.

**Bayesian Normal-Inverse-Gamma (NIG).** We treat (μ, σ²) as *uncertain parameters*
and place a weakly informative conjugate prior on them. The posterior predictive is a
Student-t distribution whose heavier tails honestly reflect the limited baseline. When
data is sparse, the NIG is more conservative (fewer false alarms); when data is dense,
it converges to the z-score. This is the methodological contribution — no published
application of conjugate NIG posterior predictive anomaly detection on SMAP
week-of-year climatology exists, to our knowledge.

## Events

Both events are configured in `configs/task3_soil_moisture.yaml` and analysed from the
*same* baseline climatology:
- **2019 Midwest flood** (April–July): record spring rainfall across IA, IL, MO.
- **2022 Great Plains flash drought** (June–August): rapid D0→D3+ escalation across KS, NE.

Analysing one wet event and one dry event from the same baseline provides
mirror-image validation of the framework.

**Outputs:** `smap_climatology.parquet` (with NIG posterior parameters), plus one
`smap_anomaly_{event_id}.parquet` per event containing both z-scores and NIG scores.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.smap_weekly_parquet import event_week_columns, load_smap_year_metadata
from src.modeling.task3_nig_anomaly import (
    nig_posterior_params,
    nig_predictive_scores,
    regional_prior_beta0,
    regional_prior_mu0,
)
from src.modeling.task3_smap_anomalies import (
    attach_cdl_year,
    baseline_climatology_iso_weeks,
    compute_event_anomalies,
    event_windows_from_cfg,
)

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

b0, b1 = int(cfg["baseline"]["period"][0]), int(cfg["baseline"]["period"][1])
events = event_windows_from_cfg(cfg)
print("Configured events:", [e["id"] for e in events])

panel_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "task3_pixel_panel.parquet"
if not panel_pq.is_file():
    raise FileNotFoundError(f"Run notebook 01 first: missing {panel_pq}")
pixels = pd.read_parquet(panel_pq)

iso_union: set[int] = set()
specs_by_id: dict[str, list] = {}
for ev in events:
    y = int(ev["year"])
    meta_y = load_smap_year_metadata(REPO_ROOT, y)
    specs = event_week_columns(meta_y, str(ev["start_date"]), str(ev["end_date"]))
    specs_by_id[str(ev["id"])] = specs
    iso_union.update(w for (_, _, w) in specs)
    print(ev["id"], "year", y, "week columns", len(specs))

iso_weeks = sorted(iso_union)
print("Union ISO weeks for climatology:", iso_weeks[0], "…", iso_weeks[-1], "count", len(iso_weeks))

# --- Frequentist climatology (mu, sigma, count) ---
clim = baseline_climatology_iso_weeks(
    REPO_ROOT, range(b0, b1 + 1), iso_weeks, pixels[["iy", "ix"]]
)

# --- NIG conjugate Bayesian posterior params per (pixel, iso_week) ---
LAMBDA_0 = 1.0
ALPHA_0 = 2.0

mu0_vec = regional_prior_mu0(clim)
beta0_vec = regional_prior_beta0(clim, alpha_0=ALPHA_0)

mu_n, lam_n, alpha_n, beta_n = nig_posterior_params(
    clim["sm_mean"].to_numpy(),
    clim["sm_std"].to_numpy(),
    clim["sm_count"].to_numpy(),
    mu_0=mu0_vec.to_numpy(),
    lam_0=LAMBDA_0,
    alpha_0=ALPHA_0,
    beta_0=beta0_vec.to_numpy(),
)

clim["nig_mu_n"] = mu_n.astype(np.float32)
clim["nig_lam_n"] = lam_n.astype(np.float32)
clim["nig_alpha_n"] = alpha_n.astype(np.float32)
clim["nig_beta_n"] = beta_n.astype(np.float32)

print(f"NIG posterior params: median df = {np.nanmedian(2 * alpha_n):.1f}, "
      f"median scale = {np.nanmedian(np.sqrt(beta_n * (1 + 1/lam_n) / alpha_n)):.4f}")

out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
clim_pq = out_dir / "smap_climatology.parquet"
clim.to_parquet(clim_pq, index=False)
print("Wrote", clim_pq.relative_to(REPO_ROOT), "rows", len(clim))

# --- Per-event anomaly Parquets: frequentist z-score + NIG Bayesian scores ---
for ev in events:
    eid = str(ev["id"])
    y = int(ev["year"])
    specs = specs_by_id[eid]

    ev_pixels = attach_cdl_year(REPO_ROOT, pixels[["iy", "ix"]], y)
    anom = compute_event_anomalies(REPO_ROOT, y, specs, clim, ev_pixels)
    anom["event_id"] = eid
    anom["event_label"] = str(ev.get("label", eid))

    p_anom, p_drought, p_scale, p_df = nig_predictive_scores(
        anom["sm_obs"].to_numpy(),
        anom["nig_mu_n"].to_numpy(),
        anom["nig_lam_n"].to_numpy(),
        anom["nig_alpha_n"].to_numpy(),
        anom["nig_beta_n"].to_numpy(),
    )

    anom["nig_p_anomaly"] = p_anom
    anom["nig_p_drought"] = p_drought
    anom["nig_posterior_scale"] = p_scale
    anom["nig_df"] = p_df

    anom_pq = out_dir / f"smap_anomaly_{eid}.parquet"
    anom.to_parquet(anom_pq, index=False)
    print(f"Wrote {anom_pq.relative_to(REPO_ROOT)}  rows={len(anom)}  "
          f"median(nig_p_drought)={np.nanmedian(p_drought):.3f}  "
          f"frac(p_anom<0.05)={np.nanmean(p_anom < 0.05):.3f}")


Configured events: ['midwest_flood_2019', 'plains_drought_2022']
midwest_flood_2019 year 2019 week columns 18
plains_drought_2022 year 2022 week columns 13
Union ISO weeks for climatology: 14 … 35 count 22
NIG posterior params: median df = 11.0, median scale = 0.0454
Wrote data\processed\task3\smap_climatology.parquet rows 45850464
Wrote data\processed\task3\smap_anomaly_midwest_flood_2019.parquet  rows=37514016  median(nig_p_drought)=0.817  frac(p_anom<0.05)=0.002
Wrote data\processed\task3\smap_anomaly_plains_drought_2022.parquet  rows=27093456  median(nig_p_drought)=0.292  frac(p_anom<0.05)=0.049
